# EXP-002 single-doc 하락 원인 분석

목적: EXP-002(BM25 문서에 메타데이터 6개 필드 접두)가 dev-full Recall@5를 EXP-001보다 −2.62%p 떨어뜨린 원인을 본다(판정 기각, `docs/experiments/EXP-002.md`). 판정과 EXP 파라미터는 바꾸지 않는 관찰이다. 질의 원문은 출력하지 않는다(D-10).

추정 원인(판정 보고): 같은 회의의 청크가 모두 같은 메타데이터, 특히 긴 `agenda`를 받아 질의 주제어가 안건과 겹치면 같은 회의 청크들이 BM25 상위를 차지한다. 하락이 소위원회에 몰렸다(−6.92%p).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from kiwipiepy import Kiwi

from rag.bm25 import BM25Retriever, forms
from rag.eval import metrics as m1
from rag.index import load_cfg
from rag.search import rank_docs, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/runs"
INDEX_DIR = ROOT / "data/index/kure-v1-fixed-512-64/full"
RUN = {
    "exp001": RUNS / "kure-v1-fixed-512-64-hybrid-dev-full.csv",
    "exp002": RUNS / "kure-v1-fixed-512-64-hybrid-meta-dev-full.csv",
}
CFG = {
    "exp001": load_cfg(["+exp=exp001", "index.scope=full"]),
    "exp002": load_cfg(["+exp=exp002", "index.scope=full"]),
}
FIELDS = list(CFG["exp002"].retriever.bm25.meta_fields)
TAGS = set(CFG["exp002"].retriever.bm25.tags)
K = 5
DEPTH = 100  # BM25 단독 순위를 보는 깊이

In [3]:
def top_k(path, k):
    run = pd.read_csv(path, dtype={"qid": str, "doc_id": str}).sort_values(["qid", "rank"])
    return run.groupby("qid")["doc_id"].agg(lambda s: list(s)[:k])


def relation(doc, gold_doc):
    if doc == gold_doc:
        return "정답"
    if meta.at[doc, "conference_number"] == meta.at[gold_doc, "conference_number"]:
        return "정답과 같은 회의"
    if meta.at[doc, "committee_name"] == meta.at[gold_doc, "committee_name"]:
        return "정답과 같은 위원회"
    return "그 밖"

## 데이터

In [4]:
cfg = CFG["exp001"]
gold = m1.load_gold(cfg, "dev-full")
top = {name: top_k(path, K)[gold.index] for name, path in RUN.items()}
with (ROOT / cfg.paths.corpus).open(encoding="utf-8") as f:
    meta = pd.DataFrame(
        [{k: d[k] for k in ["doc_id", "conference_number", *FIELDS]} for d in map(json.loads, f)]
    ).set_index("doc_id")
gold = gold.join(
    meta[["conference_number"]].rename(columns={"conference_number": "conf"}), on="doc_id"
)
gold.shape, meta.shape

((3016, 5), (38516, 7))

## 1. 회의구분별 상위 5위 변화

EXP-001에서 정답이 5위 안에 있다가 EXP-002에서 빠진 질의(이탈)와 그 반대(진입).

In [5]:
hit = {
    name: pd.Series(
        [d in t for d, t in zip(gold["doc_id"], top[name], strict=True)], index=gold.index
    )
    for name in RUN
}
labels = {
    (True, False): "이탈",
    (False, True): "진입",
    (True, True): "유지",
    (False, False): "둘 다 실패",
}
gold["change"] = [labels[(a, b)] for a, b in zip(hit["exp001"], hit["exp002"], strict=True)]
pd.crosstab(gold["meeting_name"], gold["change"], margins=True)

change,둘 다 실패,유지,이탈,진입,All
meeting_name,,,,,
국정감사,83,1131,3,8,1225
본회의,13,96,3,0,112
소위원회,81,1033,92,8,1214
예산결산특별위원회,18,138,1,1,158
특별위원회,36,268,0,3,307
All,231,2666,99,20,3016


## 2. 이탈 질의에서 올라온 문서의 정체

EXP-002 상위 5에 새로 들어온 문서를 정답과의 관계로 나눈다. 같은 회의 문서는 메타데이터가 정답과 같다.

In [6]:
ORDER = ["정답과 같은 회의", "정답과 같은 위원회", "그 밖"]
rows = []
for qid, q in gold[gold["change"] == "이탈"].iterrows():
    old, new = top["exp001"][qid], top["exp002"][qid]
    rows += [
        {"meeting_name": q["meeting_name"], "rel": relation(x, q["doc_id"])}
        for x in new
        if x not in old
    ]
promoted = pd.DataFrame(rows)
pd.crosstab(promoted["meeting_name"], promoted["rel"], margins=True).reindex(
    columns=[*ORDER, "All"], fill_value=0
)

rel,정답과 같은 회의,정답과 같은 위원회,그 밖,All
meeting_name,,,,
국정감사,0,0,3,3
본회의,0,1,2,3
소위원회,0,14,188,202
예산결산특별위원회,0,0,1,1
All,0,15,194,209


### 회의당 문서 수

같은 메타데이터를 공유하는 문서가 많을수록 그 메타데이터가 질의와 겹칠 때 함께 올라올 문서가 많다.

In [7]:
docs_per_conf = meta.groupby("conference_number").size()
conf_meeting = meta.groupby("conference_number")["meeting_name"].first()
pd.DataFrame(
    {
        "회의 수": conf_meeting.value_counts(),
        "회의당 문서 수(중앙값)": docs_per_conf.groupby(conf_meeting).median(),
        "회의당 문서 수(평균)": docs_per_conf.groupby(conf_meeting).mean().round(1),
    }
)

,회의 수,회의당 문서 수(중앙값),회의당 문서 수(평균)
meeting_name,,,
국정감사,4100,1.0,3.9
본회의,281,1.0,4.1
소위원회,3980,1.0,3.9
예산결산특별위원회,420,1.0,4.9
특별위원회,881,1.0,4.4


## 3. 메타데이터 길이

필드별 형태소 수(BM25와 같은 Kiwi 태그). 청크 본문은 512 토큰(KURE 토크나이저)이고 Kiwi 형태소로 약 500개다(EXP-001 측정 1,000청크 평균 496). 접두가 길면 BM25 문서 길이와 단어 빈도가 같이 바뀐다.

In [8]:
kiwi = Kiwi()
n_tok = {
    f: pd.Series(
        [len(t) for t in forms(kiwi, meta[f].fillna("").astype(str), TAGS)], index=meta.index
    )
    for f in FIELDS
}
n_tok = pd.DataFrame(n_tok)
n_tok["합계"] = n_tok.sum(axis=1)
n_tok.groupby(meta["meeting_name"]).median().round(1)

,date,committee_name,meeting_name,meeting_number,session_number,agenda,합계
meeting_name,,,,,,,
국정감사,7.0,3.0,2.0,1.0,6.0,2.0,21.0
본회의,7.0,2.0,1.0,2.0,2.0,5.0,19.0
소위원회,7.0,7.0,1.0,2.0,2.0,71.0,90.0
예산결산특별위원회,7.0,4.0,4.0,2.0,2.0,21.0,40.0
특별위원회,7.0,6.0,2.0,2.0,2.0,8.0,30.0


## 4. 이탈 질의의 형태소가 어느 필드에 걸리는가

이탈 질의의 형태소 중 (a) 정답 문서 메타데이터 필드에 나오는 비율, (b) 올라온 문서 메타데이터에 나오는 비율을 필드별로 본다. 올라온 문서의 안건에 질의 형태소가 많이 걸리면 `agenda`가 순위를 흔든 것이다.

In [9]:
meta_tok = {
    f: dict(
        zip(meta.index, map(set, forms(kiwi, meta[f].fillna("").astype(str), TAGS)), strict=True)
    )
    for f in FIELDS
}
lost = gold[gold["change"] == "이탈"]
q_tok = dict(
    zip(
        lost.index,
        map(set, forms(kiwi, read_texts(ROOT / cfg.paths.queries, lost.index.tolist()), TAGS)),
        strict=True,
    )
)
rows = []
for qid, q in lost.iterrows():
    qt = q_tok[qid]
    new = [x for x in top["exp002"][qid] if x not in top["exp001"][qid]]
    for f in FIELDS:
        rows.append(
            {
                "field": f,
                "group": "정답 문서",
                "share": len(qt & meta_tok[f][q["doc_id"]]) / max(len(qt), 1),
            }
        )
        rows += [
            {
                "field": f,
                "group": "올라온 문서",
                "share": len(qt & meta_tok[f][x]) / max(len(qt), 1),
            }
            for x in new
        ]
pd.DataFrame(rows).pivot_table(
    index="field", columns="group", values="share", aggfunc="mean"
).reindex(FIELDS).round(3)

group,올라온 문서,정답 문서
field,,
date,0.006,0.007
committee_name,0.013,0.009
meeting_name,0.005,0.003
meeting_number,0.000,0.000
session_number,0.004,0.001
agenda,0.038,0.065


## 5. BM25 단독 순위

RRF 결합 전 BM25 단독에서 정답 문서 순위가 어떻게 바뀌었는지 본다. 두 BM25 인덱스를 차례로 읽는다(각 약 1\~2분).

In [10]:
chunk_doc_ids = np.array(
    [json.loads(line)["doc_id"] for line in (INDEX_DIR / "chunks.jsonl").open(encoding="utf-8")]
)
texts = read_texts(ROOT / cfg.paths.queries, gold.index.tolist())
bm25_rank = {}
for name, c in CFG.items():
    bm = BM25Retriever(c, INDEX_DIR)
    ids = bm.query_ids(texts)
    ranks = []
    for i, g in zip(ids, gold["doc_id"], strict=True):
        docs = rank_docs(bm.scores(i), chunk_doc_ids, DEPTH, c.retriever.chunk_pool)
        ranks.append(docs.index(g) + 1 if g in docs else np.nan)
    bm25_rank[name] = pd.Series(ranks, index=gold.index)
    del bm
bm25 = pd.DataFrame(bm25_rank)
pd.DataFrame({f"BM25 단독 R@{k}": (bm25 <= k).mean() for k in (1, 5, 10, 100)}).T.round(4)

,exp001,exp002
BM25 단독 R@1,0.8163,0.7473
BM25 단독 R@5,0.9174,0.8597
BM25 단독 R@10,0.9416,0.8859
BM25 단독 R@100,0.9808,0.9489


In [11]:
(bm25 <= K).groupby(gold["meeting_name"]).mean().assign(
    n=gold["meeting_name"].value_counts()
).round(4)

,exp001,exp002,n
meeting_name,,,
국정감사,0.9388,0.9380,1225
본회의,0.8839,0.9196,112
소위원회,0.9086,0.7661,1214
예산결산특별위원회,0.8987,0.8861,158
특별위원회,0.8893,0.8827,307


In [12]:
pd.crosstab(
    gold["change"],
    [(bm25["exp001"] <= K).rename("BM25 exp001@5"), (bm25["exp002"] <= K).rename("BM25 exp002@5")],
)

BM25 exp001@5 False       True       
BM25 exp002@5 False True  False True 
change                               
둘 다 실패          142     3     9    77
유지               54    22   116  2474
이탈               18     0    76     5
진입                7     3     1     9

## 6. 길이 정규화 확인

2절에서 올라온 문서가 같은 회의가 아니라 대부분 "그 밖"이면, 경쟁 문서가 올라왔다기보다 정답 문서가 내려갔을 수 있다. BM25(b 0.75)는 문서가 길수록 단어 점수를 낮춘다. 소위원회 문서는 안건 접두가 길어(3절) 본문 단어 점수가 다른 문서보다 더 깎인다. 이탈 질의의 정답 문서와 올라온 문서의 회의구분·접두 길이를 비교한다.

In [13]:
rows = []
for qid, q in lost.iterrows():
    rows.append(
        {
            "group": "정답 문서",
            "meeting_name": meta.at[q["doc_id"], "meeting_name"],
            "meta_tok": n_tok.at[q["doc_id"], "합계"],
        }
    )
    rows += [
        {
            "group": "올라온 문서",
            "meeting_name": meta.at[x, "meeting_name"],
            "meta_tok": n_tok.at[x, "합계"],
        }
        for x in top["exp002"][qid]
        if x not in top["exp001"][qid]
    ]
lp = pd.DataFrame(rows)
pd.concat(
    {
        "회의구분 비율": pd.crosstab(lp["group"], lp["meeting_name"], normalize="index").round(3).T,
        "접두 형태소 수(중앙값)": lp.groupby(["meeting_name", "group"])["meta_tok"]
        .median()
        .unstack(),
    },
    axis=1,
)

회의구분 비율        접두 형태소 수(중앙값)        
group         올라온 문서  정답 문서        올라온 문서   정답 문서
meeting_name                                     
국정감사           0.335  0.030          21.0    21.0
본회의            0.077  0.030          17.0    19.0
소위원회           0.431  0.929          49.5  1106.0
예산결산특별위원회      0.067  0.010          40.0    41.0
특별위원회          0.091  0.000          28.0     NaN

In [14]:
lp.groupby("group")["meta_tok"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
올라온 문서,209.0,84.6,315.0,17.0,21.0,28.0,48.0,4166.0
정답 문서,99.0,1112.9,886.6,17.0,457.0,1104.0,1285.0,3727.0
